In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModelForCausalLM
import torch

base_model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-j-6B", device_map="auto", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-j-6B")
added_tokens = tokenizer.add_special_tokens({"bos_token": "<s>", "eos_token": "</s>", "pad_token": "<pad>"})
base_model.resize_token_embeddings(len(tokenizer))

adapter_dir = "/mnt/gpt4all_output_gptj_lora_20230515/final"
model = PeftModelForCausalLM.from_pretrained(base_model, adapter_dir, device_map="auto", torch_dtype=torch.float16)
model.to(dtype=torch.float16)


/root/anaconda3/envs/gpt4all/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPTJForCausalLM(
      (transformer): GPTJModel(
        (wte): Embedding(50403, 4096)
        (drop): Dropout(p=0.0, inplace=False)
        (h): ModuleList(
          (0-27): 28 x GPTJBlock(
            (ln_1): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
            (attn): GPTJAttention(
              (attn_dropout): Dropout(p=0.0, inplace=False)
              (resid_dropout): Dropout(p=0.0, inplace=False)
              (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
              (v_proj): Linear(
                in_features=4096, out_features=4096, bias=False
                (lora_dropout): Dropout(p=0.1, inplace=False)
                (lora_A): Linear(in_features=4096, out_features=8, bias=False)
                (lora_B): Linear(in_features=8, out_features=4096, bias=False)
              )
              (q_proj): Linear(
                in_features=4096, out_features=4096, bias=False
       

In [2]:
print(f"Mem needed: {model.get_memory_footprint() / 1024 / 1024 / 1024:.2f} GB")
print(model.device)

Mem needed: 11.39 GB
cuda


In [16]:
prompt = "Hellow please introduce yourself" + \
""".I'm an AI language model named "NLP-AI" and I'm here to assist you in generating high-quality, human-like responses to your queries.""" + \
"what is the best backend framework?"
print(prompt)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
outputs = model.generate(input_ids=input_ids, max_new_tokens=512, temperature=1)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(decoded[len(prompt):])


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hellow please introduce yourself.I'm an AI language model named "NLP-AI" and I'm here to assist you in generating high-quality, human-like responses to your queries.what is the best backend framework?

As an AI language model, I do not have a preference for a specific backend framework. However, some popular options include Python, Java, and Node.js. It ultimately depends on your specific needs and preferences.
